# 추천 시스템: 순위
---

1. 검색 단계: 가능한 모든 후보 중 수백개의 초기 후보세트를 선택합니다. 주요목적은 사용자가 관심없는 모든 후보를 효율적으로 제거하는 것입니다. 계산효율성이 좋아야한다.
2. 순위 단계:검색 모델의 출력을 가져와 미세조정하여 가능한 한 최고의 권장사항을 선택합니다. 그 작업은 사용자가 관심을 가질 만한 항목 집합을 가능한 후보의 최종 목록으로 좁히는 것입니다.

In [44]:
import os
import pprint
import tempfile

from typing import Dict, Text

import numpy as np
import tensorflow as tf
import tensorflow_datasets as tfds

import tensorflow_recommenders as tfrs

In [45]:
ratings = tfds.load("movielens/100k-ratings", split="train")
# 이번에는 ratings만 가져오네.


ratings = ratings.map(lambda x: {
    "movie_title": x["movie_title"],
    "user_id": x["user_id"],
    "user_rating": x["user_rating"]
})

for x in ratings.take(1).as_numpy_iterator():
    pprint.pprint(x)

{'movie_title': b"One Flew Over the Cuckoo's Nest (1975)",
 'user_id': b'138',
 'user_rating': 4.0}


2026-03-12 18:33:21.593897: W tensorflow/core/kernels/data/cache_dataset_ops.cc:858] The calling iterator did not fully read the dataset being cached. In order to avoid unexpected truncation of the dataset, the partially cached contents of the dataset  will be discarded. This can happen if you have an input pipeline similar to `dataset.cache().take(k).repeat()`. You should use `dataset.take(k).cache().repeat()` instead.


In [46]:
print(len(ratings))

100000


In [47]:
tf.random.set_seed(42)
shuffled = ratings.shuffle(100_000, seed=42, reshuffle_each_iteration=False)

trian = shuffled.take(80_000)
test = shuffled.skip(80_000).take(20_000)

In [48]:
# 데이터에 있는 고유한 user_id와 movie_title
movie_titles = ratings.batch(1_000_000).map(lambda x: x["movie_title"])
user_ids = ratings.batch(1_000_000).map(lambda x: x["user_id"])

unique_movie_titles = np.unique(np.concatenate(list(movie_titles)))
unique_user_ids = np.unique(np.concatenate(list(user_ids)))

unique_movie_titles[:5]

array([b"'Til There Was You (1997)", b'1-900 (1994)',
       b'101 Dalmatians (1996)', b'12 Angry Men (1957)', b'187 (1997)'],
      dtype=object)

---

리스트와 배치의 차이

.batch(1000) - 하는 이유는 기본적으로 배치 안 쓰면 하나씩 뽑아쓰게됨. 크게 배치수를하면 한번에 가져올 수 있겠지. 그리고 데이터셋에서 텐서상자가 씌어진 형태가 된다.
> Tensor(1000개) → Tensor(1000개) → Tensor(1000개)

list(movie_titles) - 상자안의 상자
> [ Tensor(1000개) → Tensor(1000개) → Tensor(1000개) ]

np.concatenate(...) - 하나의 긴 줄로 이어붙임. (리스트도 배치 벽도 허묾. 결국 하나의 거대한 넘파이 배열이 탄생)

> [ 3000개 짜리 ] 

---
**데이터셋은 {키: 값}딕셔너리를 한 번에 한 줄씩 줄줄이 흐르게 하는 장치**

    사실 딕셔너리가아닌 텐서들이 들어가있다. 딕셔너리처럼보이는텐서들의 행렬임.

- 데이터셋-> 리스트 묶기 (너무 느리미. 왜냐면 데이터셋은 한번에 한줄이니까..데이터가 너무많으면..으악!)
- 데이터셋-> 배치로 묶기 -> 리스트로 묶기 (정석)

---
10000개 있는 데이터를 

만약 배치를1000으로 하면 shape=(1000,) 이렇게 나온다. shape은 철저히 하나상자의 모형을 알려줌.

배치를 안 했을 때는 상자가 없으니 걍 ()

In [ ]:
# 지금까지 데이터셋 train/test 분할 -> 유니크 user_id, movie_title 만들기 까지도 했다.
# 이제 할 것은:: 순위 모델 구현
# 근데 아직까지 모델 만들어 놓은 게 없어서 그냥 self만 호출하는 구나
# StringLookup 레이어는 임베딩을 위해 꼭 필요한 작업임을 알고..

class RankingModel(tf.keras.Model):
    def __init__(self):
        super().__init__()
        embedding_dimension = 32 # 32개의 임베딩 숫자..
        
        # Compute embeddings for users.
        self.user_embeddings = tf.keras.Sequential([
            tf.keras.layers.StringLookup(
               vocabulary=unique_user_ids, mask_token=None), 
            # 네가 앞으로 숫자로 바꾸어야 할 단어 목록은 이거야. (user_A는 무조건 1번, user_B는 무조건 2번 !.....)
            tf.keras.layers.Embedding(len(unique_user_ids)+1, embedding_dimension)
            # Embedding(세로길이, 가로길이)
        ])
        
        # Compute embeddings for movies.
        self.movie_embeddings = tf.keras.Sequential([
            tf.keras.layers.StringLookup(
                vocabulary=unique_movie_titles, mask_token=None),
            tf.keras.layers.Embedding(len(unique_movie_titles)+1, embedding_dimension)
        ])
        
        # Compute predictions.위에 유저아이디와 영화제목이 입력값이고 이건 그 둘을 합해서 별점을 예측하는 함수.
        # 주의할 점은 32개의 임베딩 숫자를 받아서 256개-64-1개로 임베딩 수를 압축.
        self.ratings = tf.keras.Sequential([
            tf.keras.layers.Dense(256, activation="relu"),
            tf.keras.layers.Dense(64, activation="relu"),
            tf.keras.layers.Dense(1) # 최종 별점
        ])
        
    # __init__은 가구 배치만 했던 것.
    # call은 기계 돌리는 과정.
    def call(self, inputs):
        
        user_id, movie_title = inputs # 들어온 재료 박스 중에서 user_id와 movie_title을 각각꺼내.
        
        user_embedding = self.user_embeddings(user_id) # 32개 임베딩 수가 툭 튀어나옴.
        movie_embedding = self.movie_embeddings(movie_title)
        
        return self.ratings(tf.concat([user_embedding, movie_embedding], axis=1)) 
        # 임베딩 수를 길게 이어 붙여서 64개의 한 줄로 만들어.
        # 그리고 그 64개개 숫자로 이루어진 긴 줄을 ratings에 넣어서 계산해서 결과 뱉어내.
        # axis=0은 데이터를 추가할때 (밑에 쌓음)
        # axis=1은 데이터를 보강할 때. (옆에 쭈욱 붙임.)

**1. axis=0)**
- 데이터를 추가한다는 것은 사람을 늘린다는 것.
    - 기존 데이터: [나이: 20, 성별: 남]
    - 추가 데이터: [나이: 30, 성별: 여]
    - 데이터의 양을 늘릴 때 쓴다.
    
**2. aixs=1)**
- 데이터를 보강한다는 것은 정보를 늘린다는 것.
    - 기존 데이터(유저 정보): [나이: 20, 성별: 남]
    - 보강 데이터(영화 정보): [장르: 액션, 평점: 4.5]
    - 합친 결과: 1번데이터: [20, 남, 액션, 4.5] << 정보가 옆으로 길어짐

In [65]:
RankingModel()((["42"], ["One Flew Over the Cuckoo's Nest (1975)"]))
# shape(1,1)의 의미: 유저1명, 평점특징1개

<tf.Tensor: shape=(1, 1), dtype=float32, numpy=array([[-0.00017613]], dtype=float32)>

솔직히 전에 검색단계 공부했을 때랑 모델 구조차이가 은근히 많이나는 듯 한?? 

| 비교 항목 | Retrieval(검색) | Ranking(순위) |
| :--- | :--- | :--- |
| 목표 | 후보 솎아내기(TopK) | 정확한 평점 예측 |
| 평가 지표 | Tok순위 안에 있나? | RMSE/MSE 오차가 얼마인가? |


In [ ]:
# 지금까지 공부방법 구축했고 이젠 복습 방법 구축 (훈련루프를 구성하기위해 꼭필수)
task = tfrs.tasks.Ranking(
    loss = tf.keras.losses.MeanSquaredError(),
    metrics=[tf.keras.metrics.RootMeanSquaredError()]
)